In [1]:
import duckdb

# DuckDB 연결 (파일 경로가 있다면 'path/to/file.db')
con = duckdb.connect(database='/home/ai_study/projects/miniprj/data-pipeline/scripts/mimic_total.duckdb') 

In [ ]:
# 1. Chartevents 데이터 확인
query_chartevents = """
SELECT 
    itemid,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT subject_id) AS unique_patients,
    ROUND(AVG(CASE WHEN valuenum IS NULL THEN 1 ELSE 0 END) * 100, 2) AS null_percentage,
    MIN(valuenum) AS min_val,
    MAX(valuenum) AS max_val
FROM chartevents
WHERE itemid IN (
    220045, 220210, 224690, -- Heart Rate, Resp Rate
    220277, 223835,         -- O2, FiO2
    223762, 223761,         -- Temp
    220179, 220180, 220181, -- NBP
    220050, 220051, 220052, -- ABP
    220739, 223900, 223901, 226755 -- GCS
)
GROUP BY itemid
ORDER BY total_rows DESC;
"""
df_vitals = con.execute(query_chartevents).df()
print(df_vitals)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    itemid  total_rows  unique_patients  null_percentage min_val  max_val
0   220045     8752069            65365              0.0      -1     99.5
1   220210     8636655            65302              0.0       0       99
2   220277     8567015            65304              0.0      -1      999
3   220179     5378740            64703              0.0      -2  99.0003
4   220180     5377689            64702              0.0      -2      991
5   220181     5372922            64674              0.0  -22767     9975
6   220052     3096934            31414              0.0      -1     9999
7   220050     3087686            31267              0.0      -1       99
8   220051     3087261            31274              0.0      -1      997
9   220739     2209510            65226              0.0       1        4
10  223900     2205121            65219              0.0       1        5
11  223901     2199619            65213              0.0       1        6
12  223761     2055040            6444

In [5]:
# 2. Labevents 데이터 확인 (Type Cast 적용 버전)
query_labevents = """
SELECT 
    l.itemid,
    d.label,
    d.fluid,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT l.subject_id) AS unique_patients,
    COUNT(DISTINCT l.hadm_id) AS unique_admissions,
    -- valuenum을 DOUBLE로 형변환 후 평균 계산
    ROUND(AVG(CAST(l.valuenum AS DOUBLE)), 2) AS mean_value
FROM labevents l
JOIN d_labitems d ON l.itemid = d.itemid
WHERE l.itemid IN (
    50817, 50813, 50820, 50825, -- Blood Gas
    51301, 50889, 51288,          -- Inflammation
    51265, 50912, 50971, 50983, 50885 -- Coagulation, Renal, Liver
)
GROUP BY l.itemid, d.label, d.fluid
ORDER BY unique_patients DESC;
"""

try:
    df_labs = con.execute(query_labevents).df()
    print("성공적으로 데이터를 불러왔습니다.")
    display(df_labs) # Jupyter 환경이라면 display()가 더 깔끔하게 보입니다
except Exception as e:
    print(f"쿼리 실행 중 오류 발생: {e}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

성공적으로 데이터를 불러왔습니다.


,itemid,label,fluid,total_rows,unique_patients,unique_admissions,mean_value
0,51265,Platelet Count,Blood,4214048,298654,424182,230.76
1,51301,White Blood Cells,Blood,4157284,298581,421630,8.89
2,50912,Creatinine,Blood,4319091,295065,415847,1.34
3,50971,Potassium,Blood,4149507,288179,411547,4.19
4,50983,Sodium,Blood,4111289,288058,409802,138.54
5,50885,"Bilirubin, Total",Blood,1605452,185708,203854,1.50
6,50813,Lactate,Blood,670016,136279,111558,4.21
7,50820,pH,Blood,754141,75703,87825,7.37
8,50889,C-Reactive Protein,Blood,178039,57187,33783,34.36
9,50817,Oxygen Saturation,Blood,239559,42321,36316,83.62


In [7]:
# 3. Outputevents 데이터 확인 (Type Cast 및 Label 조인)
query_outputs = """
SELECT 
    o.itemid,
    d.label,
    COUNT(*) AS row_count,
    COUNT(DISTINCT o.subject_id) AS unique_patients,
    -- 문자열 value를 숫자(DOUBLE)로 변환하여 합계 계산
    ROUND(SUM(CAST(o.value AS DOUBLE)), 2) AS total_volume,
    ROUND(AVG(CAST(o.value AS DOUBLE)), 2) AS avg_volume
FROM outputevents o
LEFT JOIN d_items d ON o.itemid = d.itemid
GROUP BY o.itemid, d.label
ORDER BY total_volume DESC
LIMIT 10;
"""

try:
    df_outputs = con.execute(query_outputs).df()
    print("Outputevents 조회가 완료되었습니다.")
    display(df_outputs)
except Exception as e:
    print(f"쿼리 실행 중 오류 발생: {e}")

Outputevents 조회가 완료되었습니다.


,itemid,label,row_count,unique_patients,total_volume,avg_volume
0,226559,Foley,3599702,50505,4.395961e+08,122.12
1,226560,Void,386902,33526,1.168898e+08,302.12
2,226561,Condom Cath,83164,4814,2.224662e+07,267.50
3,226627,OR Urine,22948,18560,1.438618e+07,626.90
4,227489,GU Irrigant/Urine Volume Out,7201,685,1.425426e+07,1979.48
5,226567,Straight Cath,27121,5680,1.411114e+07,520.30
6,227488,GU Irrigant Volume In,7199,731,1.321808e+07,1836.10
7,226588,Chest Tube #1,344639,13189,1.306570e+07,37.91
8,226633,Pre-Admission,14852,10779,9.350277e+06,629.56
9,226583,Rectal Tube,11175,2629,8.166252e+06,730.76


---

In [ ]:
# 4. 분석 코호트(Cohort) 정의 (최종 수정 버전)
# 18세 이상 성인, 첫 ICU 입원, 24시간 이상 체류 
query_cohort = """
WITH first_icu_stay AS (
    SELECT 
        subject_id, 
        stay_id,
        ROW_NUMBER() OVER (PARTITION BY subject_id ORDER BY CAST(intime AS TIMESTAMP)) as rn
    FROM icustays
)
SELECT 
    ie.subject_id, 
    ie.hadm_id, 
    ie.stay_id, 
    ie.intime, 
    ie.outtime, 
    ie.los,
    -- 모든 항목을 숫자로 변환하여 계산
    (EXTRACT(YEAR FROM CAST(ie.intime AS TIMESTAMP)) 
     - CAST(p.anchor_year AS INTEGER) 
     + CAST(p.anchor_age AS INTEGER)) AS age
FROM icustays ie
JOIN patients p ON ie.subject_id = p.subject_id
JOIN first_icu_stay f ON ie.stay_id = f.stay_id
WHERE 
    f.rn = 1 
    AND (EXTRACT(YEAR FROM CAST(ie.intime AS TIMESTAMP)) 
         - CAST(p.anchor_year AS INTEGER) 
         + CAST(p.anchor_age AS INTEGER)) >= 18
    AND CAST(ie.los AS DOUBLE) >= 1.0;
"""

try:
    df_cohort = con.execute(query_cohort).df()
    print(f"코호트 생성 완료: 총 {len(df_cohort):,}명의 환자가 포함되었습니다.")
    display(df_cohort.head())
except Exception as e:
    print(f"쿼리 실행 중 오류 발생: {e}")

코호트 생성 완료: 총 51,838명의 환자가 포함되었습니다.


,subject_id,hadm_id,stay_id,intime,outtime,los,age
0,13388935,25319012,35168921,2174-09-28 11:49:15,2174-09-29 19:00:08,1.299224537037037,57
1,13388967,24010109,36586228,2115-06-01 00:12:59,2115-06-02 13:52:50,1.5693402777777778,31
2,13389036,22824123,32714868,2146-01-21 13:41:19,2146-01-23 12:56:45,1.969050925925926,58
3,13389116,20307295,33762800,2121-11-27 17:39:08,2121-11-29 19:04:16,2.0591203703703704,56
4,13389280,27800263,38327452,2130-01-29 09:11:44,2130-01-31 20:18:05,2.4627430555555554,78


In [15]:
# 나이 및 체류시간 분포 확인
print(df_cohort[['age', 'los']].describe())

# 혹시 나이가 100세 이상으로 나오는 환자가 있는지 확인 (MIMIC 특성)
print(f"90세 이상 환자 수: {len(df_cohort[df_cohort['age'] >= 90])}")

# 중복된 stay_id가 없는지 최종 확인
assert df_cohort['stay_id'].is_unique, "중복된 stay_id가 존재합니다!"

                age
count  51838.000000
mean      64.893032
std       16.688926
min       18.000000
25%       55.000000
50%       67.000000
75%       77.000000
max      103.000000
90세 이상 환자 수: 2476


In [19]:
# 5. 레이블링 (원본 테이블 기반: procedureevents, inputevents 사용)
# 성인 + 첫 ICU 입원 + 24시간 체류 코호트를 기준
query_labeling = """
WITH cohort AS (
    SELECT 
        ie.subject_id, ie.hadm_id, ie.stay_id, 
        CAST(ie.intime AS TIMESTAMP) AS intime,
        CAST(p.dod AS TIMESTAMP) AS dod
    FROM icustays ie
    JOIN patients p ON ie.subject_id = p.subject_id
    WHERE CAST(ie.los AS DOUBLE) >= 1.0
)
SELECT 
    c.stay_id,
    -- 1. 30일 이내 사망 여부
    CASE 
        WHEN c.dod IS NOT NULL AND c.dod <= c.intime + INTERVAL 30 DAY 
        THEN 1 ELSE 0 
    END AS label_30d_mortality,

    -- 2. 24시간 이후 인공호흡기(Ventilation) 시작 여부
    -- Invasive Ventilation 관련 itemid: 225792 등
    CASE 
        WHEN EXISTS (
            SELECT 1 FROM procedureevents pr
            WHERE pr.stay_id = c.stay_id 
              AND pr.itemid IN (225792, 225468, 225477) 
              AND CAST(pr.starttime AS TIMESTAMP) > c.intime + INTERVAL 24 HOUR
        ) THEN 1 ELSE 0 
    END AS label_vent_after_24h,

    -- 3. 24시간 이후 승압제(Vasopressor) 시작 여부
    -- Norepinephrine, Dopamine 등 주요 승압제 itemid
    CASE 
        WHEN EXISTS (
            SELECT 1 FROM inputevents i
            WHERE i.stay_id = c.stay_id 
              AND i.itemid IN (221906, 221289, 221749, 222315) 
              AND CAST(i.starttime AS TIMESTAMP) > c.intime + INTERVAL 24 HOUR
        ) THEN 1 ELSE 0 
    END AS label_vaso_after_24h

FROM cohort c;
"""

try:
    df_labels = con.execute(query_labeling).df()
    print("✅ 원본 테이블 기반 레이블링 완료!")
    summary = df_labels[['label_30d_mortality', 'label_vent_after_24h', 'label_vaso_after_24h']].sum()
    print("\n--- 이벤트 발생 건수 (Positive Class) ---")
    print(summary)
    display(df_labels.head())
except Exception as e:
    print(f"❌ 쿼리 실행 중 오류 발생: {e}")
    print("\n팁: 만약 여전히 테이블 오류가 난다면, 'procedureevents'나 'inputevents' 테이블명을 확인해 보세요.")

✅ 원본 테이블 기반 레이블링 완료!

--- 이벤트 발생 건수 (Positive Class) ---
label_30d_mortality     11484
label_vent_after_24h     4913
label_vaso_after_24h    13987
dtype: int64


,stay_id,label_30d_mortality,label_vent_after_24h,label_vaso_after_24h
0,34473161,0,0,0
1,35513271,0,1,1
2,35803211,0,0,0
3,35440151,0,0,0
4,36754856,0,0,0


In [3]:
# 6. 첫 24시간 활력 징후(Vital Signs) 추출
query_features = """
SELECT 
    ce.stay_id, 
    ce.itemid,
    -- 요약 통계량 계산 (모델 입력을 위해 평균값 사용)
    AVG(CAST(ce.valuenum AS DOUBLE)) AS val_avg,
    MIN(CAST(ce.valuenum AS DOUBLE)) AS val_min,
    MAX(CAST(ce.valuenum AS DOUBLE)) AS val_max
FROM chartevents ce
JOIN icustays ie ON ce.stay_id = ie.stay_id
WHERE 
    -- 1. 시간 범위 제한 (입원 후 24시간)
    CAST(ce.charttime AS TIMESTAMP) BETWEEN CAST(ie.intime AS TIMESTAMP) 
                                        AND CAST(ie.intime AS TIMESTAMP) + INTERVAL 24 HOUR
    -- 2. 주요 Vital Sign 필터링
    -- 220045(HR), 220179(SBP), 220180(DBP), 220210(RR)
    AND ce.itemid IN (220045, 220179, 220180, 220210)
    -- 3. 비정상치 제거 (데이터 클렌징)
    AND CAST(ce.valuenum AS DOUBLE) > 0
GROUP BY ce.stay_id, ce.itemid;
"""

try:
    print("데이터 추출 중... (chartevents는 용량이 커서 시간이 조금 소요될 수 있습니다)")
    df_features = con.execute(query_features).df()
    print(f"✅ 피처 추출 완료: {df_features.shape[0]} 행")
    display(df_features.head())
except Exception as e:
    print(f"❌ 쿼리 실행 중 오류 발생: {e}")

데이터 추출 중... (chartevents는 용량이 커서 시간이 조금 소요될 수 있습니다)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ 피처 추출 완료: 361394 행


,stay_id,itemid,val_avg,val_min,val_max
0,34671743,220179,108.444444,79.0,150.0
1,31142025,220179,99.041667,78.0,118.0
2,33281088,220179,91.633333,78.0,140.0
3,39142259,220045,96.434783,85.0,119.0
4,36535245,220179,110.560000,82.0,134.0
